# 03. Simple Linear Regression: Theory and Workflow

This chapter introduces the simple linear regression model and fits one example from start to finish.

The statistical model is:

$$
Y_i = \beta_0 + \beta_1 x_i + \epsilon_i
$$

For this data set:

- response $Y$: `Fuelcons`
- predictor $x$: `Temp`
- question: how does expected fuel consumption change as temperature changes?

## What regression is doing

Regression predicts or explains a response variable using one or more predictor variables. If we ignored predictors, a rough prediction for a response would be the sample mean. Regression improves on that idea by letting the prediction change with information in $x$.

For simple linear regression, each observation is modeled as:

$$
y_i = \beta_0 + \beta_1 x_i + \epsilon_i.
$$

Here:

- $\beta_0$ is the intercept,
- $\beta_1$ is the slope,
- $\epsilon_i$ is the random error for observation $i$,
- $\hat y_i = b_0 + b_1x_i$ is the fitted value from the data.

The slope is the main object of interpretation. It is the estimated change in the mean response for a one-unit increase in the predictor.

## Why "linear" regression can include curves

"Linear" means linear in the unknown coefficients, not necessarily a perfectly straight plot in the original variable. For example, these are linear regression models:

$$
y = \beta_0 + \beta_1x + \epsilon
$$

and

$$
y = \beta_0 + \beta_1x^2 + \epsilon.
$$

The second model uses $x^2$ as a predictor, but the coefficients still enter linearly. A model such as $y = \beta_0 + e^{\beta_1x} + \epsilon$ is not linear in the same sense.

## Least squares

The fitted line is chosen by minimizing the sum of squared residuals:

$$
SSE = \sum_{i=1}^{n}(y_i-\hat y_i)^2.
$$

The least-squares estimators are:

$$
b_1 = \frac{\sum_{i=1}^{n}(x_i-\bar x)(y_i-\bar y)}{\sum_{i=1}^{n}(x_i-\bar x)^2}
$$

and

$$
b_0 = \bar y - b_1\bar x.
$$

These formulas are useful because they show the role of co-movement between $x$ and $y$. If large $x$ values tend to occur with small $y$ values, the numerator is negative and the slope is negative.

For ordinary least squares with an intercept:

- the residuals sum to zero,
- the fitted line passes through $(\bar x,\bar y)$,
- the fitted line has the smallest SSE among all straight lines.

## Load and plot the data

## Browser package setup

Run the next cell before the first analysis cell. It loads the scientific Python packages used in this module into the browser-based Python kernel.

The first run in a browser session can take a little time. Later notebooks usually load faster because the browser can reuse cached packages.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols

from checks import check_close, check_sign, model_snapshot

fuel = pd.read_csv("data/fuelcons.csv")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fuel["Temp"], fuel["Fuelcons"])
ax.set_xlabel("Temperature")
ax.set_ylabel("Fuel consumption")
ax.set_title("Fuel consumption vs. temperature")
plt.show()

## Fit the model

`statsmodels` uses formula notation:

```text
response ~ predictor
```

The formula `Fuelcons ~ Temp` means "fit fuel consumption as a linear function of temperature."

In [ ]:
model = ols("Fuelcons ~ Temp", data=fuel).fit()
print(model.summary())

## Extract the quantities we need

The full summary is useful, but in practice you should learn to pull out specific quantities.

In [ ]:
snapshot = model_snapshot(model)
snapshot

In [ ]:
intercept = snapshot["intercept"]
slope = snapshot["slope"]
r_squared = snapshot["r_squared"]
slope_p_value = snapshot["slope_p_value"]
rmse = snapshot["rmse"]

print(f"Intercept: {intercept:.3f}")
print(f"Slope: {slope:.3f}")
print(f"R-squared: {r_squared:.3f}")
print(f"Slope p-value: {slope_p_value:.4f}")
print(f"RMSE: {rmse:.3f}")

## Checkpoint 1: slope

The fitted slope should be negative. In this example, each one-unit increase in temperature is associated with a decrease of about 0.128 units in expected fuel consumption.

In [ ]:
student_slope = slope

print(check_sign("slope", student_slope, "negative"))
print(check_close(
    "slope",
    student_slope,
    expected=-0.1279,
    tolerance=0.005,
    hint="Use model.params['Temp']."
))

## Plot the fitted line

In [ ]:
fuel_with_fit = fuel.copy()
fuel_with_fit["fitted"] = model.fittedvalues

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fuel_with_fit["Temp"], fuel_with_fit["Fuelcons"], label="Observed")
ax.plot(
    fuel_with_fit["Temp"],
    fuel_with_fit["fitted"],
    color="red",
    label="Fitted line",
)
ax.set_xlabel("Temperature")
ax.set_ylabel("Fuel consumption")
ax.legend()
plt.show()

## Interpret the fitted equation

The fitted equation is:

$$
\widehat{Fuelcons} = 15.838 - 0.128 Temp
$$

Interpretation:

- Intercept: when `Temp = 0`, the fitted mean fuel consumption is 15.838. This may be outside the useful range of the data, so interpret it carefully.
- Slope: for each one-unit increase in `Temp`, the fitted mean fuel consumption decreases by about 0.128 units.
- R-squared: about 90% of the variation in fuel consumption is explained by a straight-line relationship with temperature in this small data set.

The fitted line passes through the sample center. We can verify that quickly.

In [ ]:
x_bar = fuel["Temp"].mean()
y_bar = fuel["Fuelcons"].mean()
fitted_at_x_bar = intercept + slope * x_bar

print(f"x-bar = {x_bar:.3f}")
print(f"y-bar = {y_bar:.3f}")
print(f"fitted value at x-bar = {fitted_at_x_bar:.3f}")
print(check_close("fitted value at x-bar", fitted_at_x_bar, expected=y_bar, tolerance=1e-10))

## Checkpoint 2: predicted mean at Temp = 40

Use the fitted equation to compute the predicted mean fuel consumption when `Temp = 40`.

In [ ]:
temp_value = 40
predicted_mean_40 = intercept + slope * temp_value
predicted_mean_40

In [ ]:
print(check_close(
    "predicted mean at Temp = 40",
    predicted_mean_40,
    expected=10.721,
    tolerance=0.02,
    hint="Use intercept + slope * 40."
))

## From output to conclusion

A good model conclusion is not just a number. It should mention the variables, direction, approximate magnitude, and uncertainty or limitations.

```{admonition} Reference conclusion
:class: dropdown

In this sample, fuel consumption decreases as temperature increases. The fitted model estimates a slope of about -0.128, meaning that each one-unit increase in temperature is associated with about 0.128 fewer units of expected fuel consumption. The relationship is strong in this small data set, with R-squared about 0.90, but the conclusion should be checked with residual diagnostics before using the model for decisions.
```

## What to reuse

For a new simple regression problem, change only three things:

1. the data file,
2. the response column,
3. the predictor column.

The workflow stays the same: plot, fit, extract, interpret, diagnose.